# Kiko Hey Kiko Wake Model Training

Beginner path: run these cells in order. You do not need to record your own voice. This notebook generates synthetic `Hey Kiko` positives, prepares free baseline negative/noise data, trains a local TFLite model, checks the export, and downloads `hey_kiko.tflite` plus `training_report.json`.

No Picovoice, no paid SDK, no API keys, and no cloud wake detection are used.


## Step 1: setup dependencies
Set Runtime -> Change runtime type -> GPU before running.


In [ ]:
!git clone https://github.com/SkyBotsDeveloper/Kiko.git /content/Kiko || true
%cd /content/Kiko
!git fetch origin v2-wake-word
!git checkout v2-wake-word
!apt-get update -qq
!apt-get install -y -qq espeak-ng
!python -m pip install --upgrade pip
!python -m pip install -r tools/wake_training/requirements.txt
!python - <<'PY'
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
PY


## Step 2: generate synthetic `Hey Kiko` positive samples


In [ ]:
!python tools/wake_training/generate_synthetic_positives.py --count 1000 --phrase "hey kiko" --validation-ratio 0.1


## Step 3: prepare free negative/noise data
Sanity is small and fast. Use balanced/quality later.


In [ ]:
!python tools/wake_training/prepare_free_negatives.py --profile sanity


## Step 4: run sanity training
This only verifies the pipeline; it is not expected to be accurate.


In [ ]:
!python tools/wake_training/train_hey_kiko.py --profile sanity --batch-size 8 --max-vram-gb 2.5


## Step 5: optional balanced or quality training
Run one of these after sanity succeeds.


In [ ]:
RUN_BALANCED = False
RUN_QUALITY = False

if RUN_BALANCED:
    !python tools/wake_training/prepare_free_negatives.py --profile balanced
    !python tools/wake_training/generate_synthetic_positives.py --count 3000 --phrase "hey kiko" --validation-ratio 0.1
    !python tools/wake_training/train_hey_kiko.py --profile balanced --batch-size 12 --max-vram-gb 2.5

if RUN_QUALITY:
    !python tools/wake_training/prepare_free_negatives.py --profile quality
    !python tools/wake_training/generate_synthetic_positives.py --count 6000 --phrase "hey kiko" --validation-ratio 0.1
    !python tools/wake_training/train_hey_kiko.py --profile quality --batch-size 8 --max-vram-gb 2.5 --resume


## Step 6: run export check


In [ ]:
!python tools/wake_training/export_check.py tools/wake_training/output/hey_kiko.tflite


## Step 7: download model and report
The model currently needs an Android log-mel adapter before live wake detection works.


In [ ]:
from google.colab import files
files.download('tools/wake_training/output/hey_kiko.tflite')
files.download('tools/wake_training/output/training_report.json')


## Next Android step
Install the model locally with:

```bash
python tools/wake_training/install_model_to_app.py tools/wake_training/output/hey_kiko.tflite
```

Do not commit the generated model. `*.tflite`, generated audio, and datasets are gitignored.
